# Gait Autoencoder Training — CASIA-B (Normal-Only)

Trains the `TransformerAutoencoder` on **normal gait sequences only** (nm-01..nm-04).

**Why retrain?** The existing checkpoint was trained on mixed normal+abnormal data →
near-zero separation (Δμ=0.0002). Training on normal-only data forces the autoencoder
to specialise; abnormal sequences (bg/cl) then produce higher reconstruction error.

**Target:** error_gap (abn_mean − nm_mean) > 0.005  (current baseline: 0.0002)

**Colab**: Runtime → Change runtime type → T4 GPU  
**Local**: Skip the Drive mount cell.

In [2]:
%pip install torch torchvision Pillow numpy

In [5]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted.')
else:
    print('Local mode.')

Mounted at /content/drive
Drive mounted.


In [6]:
import sys
IN_COLAB = 'google.colab' in sys.modules
print(f'Running in Colab: {IN_COLAB}')

Running in Colab: True


In [7]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # ── EDIT: path to your project on Google Drive ──
    PROJECT_ROOT = Path('/content/drive/MyDrive/majorProject')
else:
    PROJECT_ROOT = Path('..').resolve()
# PROJECT_ROOT = Path('/home/himanshu-kumar-jha/Documents/majorProject')  # change this

GAIT_DIR = PROJECT_ROOT / 'models' / 'casib-b'
DATA_DIR = PROJECT_ROOT / 'datasets' / 'casia-b'
RESULTS  = PROJECT_ROOT / 'results'

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(GAIT_DIR))

assert DATA_DIR.exists(), f'CASIA-B not found: {DATA_DIR}'
RESULTS.mkdir(exist_ok=True)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_DIR    :', DATA_DIR)

AssertionError: CASIA-B not found: /content/drive/MyDrive/majorProject/datasets/casia-b

In [ ]:
import os, csv, glob, time
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from train import TransformerAutoencoder, GaitSequenceDataset, _gauss_kernel

DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE  = 32 if DEVICE == 'cuda' else 8
NUM_WORKERS = 4  if DEVICE == 'cuda' else 0
SEQ_LEN     = 15
IMG_SIZE    = (64, 64)
LATENT_DIM  = 512

print(f'Device: {DEVICE}  |  batch={BATCH_SIZE}')
if DEVICE == 'cuda':
    print(f'GPU  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
def build_split_index(data_dir, conditions, seq_len, step=1):
    """Build list-of-frame-path-lists for specific condition names only."""
    out_dir = data_dir / 'output' if (data_dir / 'output').is_dir() else data_dir
    index = []
    for subj in sorted(os.listdir(out_dir)):
        sp = out_dir / subj
        if not sp.is_dir(): continue
        for cond in sorted(os.listdir(sp)):
            if cond not in conditions: continue
            cp = sp / cond
            if not cp.is_dir(): continue
            for angle in sorted(os.listdir(cp)):
                ap = cp / angle
                if not ap.is_dir(): continue
                frames = sorted(glob.glob(str(ap / '*.png')))
                if len(frames) < seq_len: continue
                for start in range(0, len(frames) - seq_len + 1, step):
                    index.append(frames[start:start + seq_len])
    return index

print('Building dataset indices ...')
train_idx = build_split_index(DATA_DIR, {'nm-01','nm-02','nm-03','nm-04'}, SEQ_LEN, step=1)
val_idx   = build_split_index(DATA_DIR, {'nm-05','nm-06'},                 SEQ_LEN, step=2)
abn_idx   = build_split_index(DATA_DIR, {'bg-01','bg-02','cl-01','cl-02'}, SEQ_LEN, step=4)
print(f'  Train (nm-01..04): {len(train_idx):,} sequences')
print(f'  Val   (nm-05..06): {len(val_idx):,} sequences')
print(f'  Abnormal monitor : {min(len(abn_idx),400)} sequences (NOT used in training)')

In [ ]:
def ssim_loss_tensor(recon, original, window_size=11):
    """Differentiable mean(1-SSIM) over a (B,T,C,H,W) tensor pair."""
    B, T, C, H, W = recon.shape
    kernel  = _gauss_kernel(window_size, sigma=1.5, device=recon.device).expand(C,1,-1,-1)
    padding = window_size // 2
    C1, C2  = 0.01**2, 0.03**2
    losses  = []
    for t in range(T):
        r, o   = recon[:,t], original[:,t]
        mu_r   = F.conv2d(r,   kernel, padding=padding, groups=C)
        mu_o   = F.conv2d(o,   kernel, padding=padding, groups=C)
        sig_r  = F.conv2d(r*r, kernel, padding=padding, groups=C) - mu_r**2
        sig_o  = F.conv2d(o*o, kernel, padding=padding, groups=C) - mu_o**2
        sig_ro = F.conv2d(r*o, kernel, padding=padding, groups=C) - mu_r*mu_o
        ssim   = ((2*mu_r*mu_o+C1)*(2*sig_ro+C2)) / ((mu_r**2+mu_o**2+C1)*(sig_r+sig_o+C2)+1e-8)
        losses.append(1.0 - ssim.mean())
    return torch.stack(losses).mean()

def combined_loss(recon, original):
    return 0.5*F.mse_loss(recon, original) + 0.5*ssim_loss_tensor(recon, original)

print('Loss: 0.5*MSE + 0.5*SSIM  (differentiable)')

In [ ]:
EPOCHS   = 80
PATIENCE = 15
LR       = 1e-4

train_dl = DataLoader(
    GaitSequenceDataset(train_idx, SEQ_LEN, IMG_SIZE),
    batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE=='cuda'), drop_last=True)
val_dl   = DataLoader(
    GaitSequenceDataset(val_idx, SEQ_LEN, IMG_SIZE),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

model = TransformerAutoencoder(latent_dim=LATENT_DIM, seq_len=SEQ_LEN).to(DEVICE)
opt   = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)

best_val, patience_c, log_rows = float('inf'), 0, []
ckpt = GAIT_DIR / 'best_gait_v2.pth'

def _mean_err(idx_list, max_n=400):
    ds = GaitSequenceDataset(idx_list[:max_n], SEQ_LEN, IMG_SIZE)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
    es = []
    with torch.no_grad():
        for c in dl:
            c = c.to(DEVICE)
            for b in range(c.size(0)):
                es.append(F.mse_loss(model(c)[b], c[b]).item())
    return float(np.mean(es)) if es else 0.0

print(f'Starting training — {EPOCHS} epochs, patience={PATIENCE}\n')
for epoch in range(1, EPOCHS+1):
    t0 = time.time()
    model.train()
    tr = []
    for clips in train_dl:
        clips = clips.to(DEVICE)
        loss  = combined_loss(model(clips), clips)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        tr.append(loss.item())
    tr_loss = float(np.mean(tr))

    model.eval()
    vl = []
    with torch.no_grad():
        for clips in val_dl:
            clips = clips.to(DEVICE)
            vl.append(combined_loss(model(clips), clips).item())
    vl_loss = float(np.mean(vl))
    sched.step()

    gap, gap_str = None, ''
    if epoch % 5 == 0 and abn_idx:
        model.eval()
        nm_err = _mean_err(val_idx);  abn_err = _mean_err(abn_idx)
        gap    = abn_err - nm_err
        flag   = '✓ improving' if gap > 0.005 else '← still low'
        gap_str = f'  gap={gap:+.5f} {flag}'

    log_rows.append({'epoch':epoch,'train_loss':round(tr_loss,6),
                     'val_loss':round(vl_loss,6),'error_gap':round(gap,6) if gap else ''})
    print(f'Ep {epoch:3d}/{EPOCHS}  tr={tr_loss:.5f}  vl={vl_loss:.5f}{gap_str}  [{time.time()-t0:.1f}s]')

    if vl_loss < best_val:
        best_val, patience_c = vl_loss, 0
        torch.save(model.state_dict(), ckpt)
        print(f'  → Saved {ckpt.name}')
    else:
        patience_c += 1
        if patience_c >= PATIENCE:
            print(f'Early stop at epoch {epoch}'); break

print(f'\nBest val loss : {best_val:.5f}')
print(f'Checkpoint    : {ckpt}')

In [ ]:
import csv
log_path = RESULTS / 'gait_train_log.csv'
with open(log_path, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['epoch','train_loss','val_loss','error_gap'])
    w.writeheader(); w.writerows(log_rows)
print(f'Log → {log_path}')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(RESULTS / 'gait_train_log.csv')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(df['epoch'], df['train_loss'], label='Train')
ax1.plot(df['epoch'], df['val_loss'],   label='Val')
ax1.set_xlabel('Epoch'); ax1.set_title('Reconstruction Loss'); ax1.legend()

gap_df = df[df['error_gap'].notna() & (df['error_gap'].astype(str) != '')].copy()
gap_df['error_gap'] = gap_df['error_gap'].astype(float)
if not gap_df.empty:
    ax2.plot(gap_df['epoch'], gap_df['error_gap'], 'ro-', label='abn − nm gap')
    ax2.axhline(0.005, linestyle='--', color='green', label='Target 0.005')
    ax2.axhline(0.0002, linestyle=':', color='gray',  label='Baseline 0.0002')
    ax2.set_xlabel('Epoch'); ax2.set_title('Error Gap (want > 0.005)'); ax2.legend()
plt.tight_layout()
out = RESULTS / 'gait_training_curves.png'
plt.savefig(str(out), dpi=120); plt.show()
print(f'Saved → {out}')